# Temperature Sensor Fault Detection
## Phase 5: Full Evaluation Report

---
**What this notebook produces:**
- Confusion matrix (raw counts + normalized %)
- ROC curves for all 4 classes with AUC scores
- Per-class Precision, Recall, F1 breakdown
- Cross-validation stability check
- Error analysis (where exactly the model fails)
- All-model comparison table
- Final summary printout

---
## 1. Imports & Load Everything

In [ ]:
import os, json, pickle, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_fscore_support
)
from sklearn.preprocessing   import label_binarize
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

CLASS_COLORS = {
    'normal'        : '#43A047',
    'noise_fault'   : '#E53935',
    'bias_fault'    : '#FB8C00',
    'flatline_fault': '#8E24AA',
}

print('Imports done.')

In [ ]:
# ================================================================
#  ONLY CHANGE THIS if your path is different
# ================================================================
BASE_PATH  = r'C:\PVS_Dataset'
PHASE2_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase2')
PHASE3_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase3')
PHASE4_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase4')
OUTPUT_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase5')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load configs
with open(os.path.join(PHASE2_DIR, 'temp_phase2_config.json')) as f: p2 = json.load(f)
with open(os.path.join(PHASE3_DIR, 'temp_phase3_config.json')) as f: p3 = json.load(f)
with open(os.path.join(PHASE4_DIR, 'temp_phase4_config.json')) as f: p4 = json.load(f)

CLASS_LABELS = {int(k): v for k, v in p2['CLASS_LABELS'].items()}
CLASS_NAMES  = [CLASS_LABELS[i] for i in sorted(CLASS_LABELS)]
FINAL_FEATS  = p3['FINAL_FEATS']

# Load splits
with open(p3['splits_path'],     'rb') as f: splits     = pickle.load(f)
with open(p4['all_models_path'], 'rb') as f: all_models = pickle.load(f)
with open(p3['scaler_path'],     'rb') as f: scaler     = pickle.load(f)

X_train   = splits['X_train'];   X_test   = splits['X_test']
X_train_s = splits['X_train_s']; X_test_s = splits['X_test_s']
y_train   = splits['y_train'];   y_test   = splits['y_test']

BEST_NAME  = p4['best_model_name']
best_model = all_models[BEST_NAME]
SCALED_MODELS = ['KNN (k=7)', 'SVM (RBF kernel)']
X_test_best = X_test_s if BEST_NAME in SCALED_MODELS else X_test

print(f'Best model : {BEST_NAME}')
print(f'All models : {list(all_models.keys())}')
print(f'Test set   : {X_test.shape[0]:,} samples  |  {len(CLASS_NAMES)} classes')
print(f'Classes    : {CLASS_NAMES}')

---
## 2. Predictions & Core Metrics

In [ ]:
y_pred = best_model.predict(X_test_best)
acc    = accuracy_score(y_test, y_pred)
f1_mac = f1_score(y_test, y_pred, average='macro')
f1_wtd = f1_score(y_test, y_pred, average='weighted')

print(f'EVALUATION REPORT — {BEST_NAME}')
print('=' * 60)
print(f'  Test Accuracy  : {acc*100:.4f}%')
print(f'  F1 Macro       : {f1_mac*100:.4f}%')
print(f'  F1 Weighted    : {f1_wtd*100:.4f}%')
print()
print('Per-class breakdown:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

---
## 3. Confusion Matrix

In [ ]:
cm     = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.6, ax=axes[0], cbar=False,
            annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_title(f'{BEST_NAME}\nConfusion Matrix — Raw Counts',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Class', fontsize=10)
axes[0].set_ylabel('Actual Class',    fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

# Normalized %
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.6, ax=axes[1],
            cbar_kws={'label': '% of actual class'},
            annot_kws={'size': 13, 'weight': 'bold'})
axes[1].set_title(f'{BEST_NAME}\nConfusion Matrix — Normalized (%)',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Class', fontsize=10)
axes[1].set_ylabel('Actual Class',    fontsize=10)
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('How to read: rows = actual, columns = predicted. Diagonal = correct.\n'
              'Off-diagonal = misclassifications. Normalized shows % of each actual class.',
              fontsize=9, color='gray')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_01_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_01_confusion_matrix.png')

# Print diagonal stats
print()
print('Per-class accuracy (from diagonal):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:20s}: {cm_pct[i,i]:.2f}% correctly classified')

---
## 4. ROC Curves

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3])

if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test_best)
elif hasattr(best_model, 'decision_function'):
    y_prob = best_model.decision_function(X_test_best)
    # Normalize to [0,1] for multi-class
    y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min() + 1e-9)
else:
    print('Model does not support probability output. Skipping ROC.')
    y_prob = None

if y_prob is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    auc_scores = {}
    # Left: all 4 ROC curves overlaid
    for i, (cls_id, cls_name) in enumerate(CLASS_LABELS.items()):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        auc = roc_auc_score(y_test_bin[:, i], y_prob[:, i])
        auc_scores[cls_name] = auc
        axes[0].plot(fpr, tpr, lw=2.5,
                     color=list(CLASS_COLORS.values())[i],
                     label=f'{cls_name}  (AUC = {auc:.4f})')

    axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6,
                  label='Random classifier (AUC = 0.50)')
    macro_auc = roc_auc_score(y_test_bin, y_prob,
                               average='macro', multi_class='ovr')
    axes[0].set_title(f'ROC Curves — {BEST_NAME}\nMacro-average AUC = {macro_auc:.4f}',
                       fontsize=12, fontweight='bold')
    axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
    axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)')
    axes[0].legend(fontsize=9, loc='lower right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.02])

    # Right: AUC bar chart per class
    names  = list(auc_scores.keys())
    aucs   = list(auc_scores.values())
    colors = [CLASS_COLORS[n] for n in names]
    bars   = axes[1].bar(names, aucs, color=colors, alpha=0.85,
                          edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, aucs):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                      val + 0.002, f'{val:.4f}',
                      ha='center', fontsize=10, fontweight='bold')
    axes[1].axhline(1.0,  color='green', lw=1.2, linestyle='--', alpha=0.5, label='Perfect (1.0)')
    axes[1].axhline(0.5,  color='red',   lw=1.2, linestyle='--', alpha=0.5, label='Random  (0.5)')
    axes[1].set_title('AUC Score per Class', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('AUC Score')
    axes[1].set_ylim([0, 1.08])
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'eval_02_roc_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: eval_02_roc_curves.png')

    print()
    print(f'Macro-average AUC : {macro_auc:.4f}')
    for name, auc in auc_scores.items():
        print(f'  {name:20s}: AUC = {auc:.4f}')

---
## 5. Per-Class Precision, Recall & F1

In [ ]:
prec, rec, f1s, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1, 2, 3])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar chart
x = np.arange(4); w = 0.25
b1 = axes[0].bar(x - w, prec*100, w, label='Precision', color='#1565C0', alpha=0.85)
b2 = axes[0].bar(x,     rec *100, w, label='Recall',    color='#2E7D32', alpha=0.85)
b3 = axes[0].bar(x + w, f1s *100, w, label='F1 Score',  color='#E65100', alpha=0.85)
for i in range(4):
    axes[0].text(i-w, prec[i]*100+0.4, f'{prec[i]*100:.1f}', ha='center', fontsize=8)
    axes[0].text(i,   rec[i] *100+0.4, f'{rec[i] *100:.1f}', ha='center', fontsize=8)
    axes[0].text(i+w, f1s[i] *100+0.4, f'{f1s[i] *100:.1f}', ha='center', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_NAMES, rotation=15, fontsize=9)
axes[0].set_ylabel('Score (%)')
axes[0].set_ylim([0, 110])
axes[0].set_title(f'Per-Class Metrics — {BEST_NAME}', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Support (sample count per class in test set)
colors = [list(CLASS_COLORS.values())[i] for i in range(4)]
bars = axes[1].bar(CLASS_NAMES, sup, color=colors, alpha=0.85,
                    edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, sup):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                  val + sup.max()*0.01,
                  f'{val:,}', ha='center', fontsize=10)
axes[1].set_title('Test Set — Samples per Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Test Windows')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_03_per_class_metrics.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_03_per_class_metrics.png')

# Table
prf_df = pd.DataFrame({
    'Class'    : CLASS_NAMES,
    'Precision': [f'{v*100:.2f}%' for v in prec],
    'Recall'   : [f'{v*100:.2f}%' for v in rec],
    'F1 Score' : [f'{v*100:.2f}%' for v in f1s],
    'Support'  : sup,
})
print()
display(prf_df)

---
## 6. Cross-Validation Stability

In [ ]:
# 5-fold cross-validation on the training set
# This tells us if the model is stable or just got lucky on this split
print(f'Running 5-fold cross-validation on {BEST_NAME}...')
print('(This may take a minute)')

X_cv = X_train_s if BEST_NAME in SCALED_MODELS else X_train
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(best_model, X_cv, y_train, cv=cv,
                          scoring='accuracy', n_jobs=-1)
cv_f1  = cross_val_score(best_model, X_cv, y_train, cv=cv,
                          scoring='f1_macro', n_jobs=-1)

print(f'\nCross-Validation Results (5-fold):')
print(f'  Accuracy : {cv_acc.mean()*100:.3f}% +/- {cv_acc.std()*100:.3f}%')
print(f'  F1 Macro : {cv_f1.mean()*100:.3f}% +/- {cv_f1.std()*100:.3f}%')
print(f'  Per-fold F1: {[f"{v*100:.2f}" for v in cv_f1]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, metric in [
    (axes[0], cv_acc*100, 'Accuracy'),
    (axes[1], cv_f1 *100, 'F1 Macro'),
]:
    folds = [f'Fold {i+1}' for i in range(5)]
    bars  = ax.bar(folds, scores, color='#1565C0', alpha=0.8,
                    edgecolor='black', lw=0.5)
    ax.axhline(scores.mean(), color='red', lw=2, linestyle='--',
                label=f'Mean = {scores.mean():.2f}%')
    ax.axhspan(scores.mean()-scores.std(),
                scores.mean()+scores.std(),
                alpha=0.1, color='red', label=f'+/- 1 std ({scores.std():.2f}%)')
    for bar, val in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2,
                 val+0.1, f'{val:.2f}%', ha='center', fontsize=9)
    ax.set_title(f'5-Fold CV {metric} — {BEST_NAME}',
                  fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{metric} (%)')
    ax.set_ylim([max(0, scores.min()-3), 101])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_04_cross_validation.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_04_cross_validation.png')

if cv_f1.std() < 0.01:
    print('Model is very stable across folds — low variance in performance.')
elif cv_f1.std() < 0.03:
    print('Model is moderately stable.')
else:
    print('High variance across folds — model performance depends heavily on which data it sees.')

---
## 7. Error Analysis — Where Does the Model Fail?

In [ ]:
errors   = y_test != y_pred
n_errors = int(errors.sum())

print(f'Total misclassifications: {n_errors:,} / {len(y_test):,}  ({n_errors/len(y_test)*100:.3f}%)')
print()

if n_errors == 0:
    print('Model made zero errors on the test set.')
    print()
    print('What this likely means:')
    print('  1. The injected faults are too easy (amplitudes too large vs signal noise)')
    print('  2. The feature set (especially unique_ratio, max_diff, trend) perfectly')
    print('     separates the 4 classes — no ambiguity in feature space')
    print()')
    print('What to write in your report:')
    print('  Acknowledge this as a limitation of synthetic fault injection.')
    print('  Real sensors produce subtle faults that overlap with normal variation.')
    print('  The 100% result establishes an upper bound — in practice, expect 80-95%.')
else:
    error_pairs = {}
    for actual, predicted in zip(y_test[errors], y_pred[errors]):
        key = (CLASS_LABELS[actual], CLASS_LABELS[predicted])
        error_pairs[key] = error_pairs.get(key, 0) + 1

    print('Misclassification breakdown:')
    print(f'{"Actual":20s} --> {"Predicted":20s}  Count   % of errors')
    print('-'*65)
    for (actual, predicted), cnt in sorted(error_pairs.items(), key=lambda x: -x[1]):
        pct = cnt / n_errors * 100
        print(f'  {actual:20s} --> {predicted:20s}  {cnt:>5,}   ({pct:.1f}%)')

    # Visualize error heatmap
    error_matrix = np.zeros((4, 4), dtype=int)
    for (act, pred), cnt in error_pairs.items():
        i = CLASS_NAMES.index(act)
        j = CLASS_NAMES.index(pred)
        error_matrix[i, j] = cnt

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(error_matrix, annot=True, fmt='d', cmap='Reds',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax, annot_kws={'size': 12})
    ax.set_title(f'Misclassification Heatmap — {BEST_NAME}\n'
                  f'(0 on diagonal = no errors for that class)',
                  fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted (wrong)'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'eval_05_error_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: eval_05_error_heatmap.png')

---
## 8. All-Model Comparison

In [ ]:
import time
print('Evaluating all models...')

model_results = []
for name, model in all_models.items():
    X_te = X_test_s if name in SCALED_MODELS else X_test
    t0   = time.time()
    yp   = model.predict(X_te)
    ms   = (time.time()-t0)/len(X_te)*1000
    model_results.append({
        'Model'          : name,
        'Accuracy (%)'   : round(accuracy_score(y_test,yp)*100, 3),
        'F1 Macro (%)'   : round(f1_score(y_test,yp,average='macro')*100, 3),
        'F1 Weighted (%)': round(f1_score(y_test,yp,average='weighted')*100, 3),
        'Infer (ms/smpl)': round(ms, 4),
    })

comp_df = pd.DataFrame(model_results).sort_values('F1 Macro (%)', ascending=False)
comp_df = comp_df.reset_index(drop=True)
comp_df.index = comp_df.index + 1
comp_df.insert(0, 'Rank', comp_df.index)
print()
display(comp_df)

# Bar chart comparison
n    = len(comp_df)
x    = np.arange(n)
w    = 0.28
cols = ['#1565C0','#2E7D32','#E65100','#6A1B9A'][:n]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(x-w/2, comp_df['Accuracy (%)'],  w, label='Accuracy',  color='#1565C0', alpha=0.85)
axes[0].bar(x+w/2, comp_df['F1 Macro (%)'],  w, label='F1 Macro',  color='#2E7D32', alpha=0.85)
for i,(acc,f1m) in enumerate(zip(comp_df['Accuracy (%)'],comp_df['F1 Macro (%)'])):
    axes[0].text(i-w/2, acc+0.2, f'{acc:.1f}', ha='center', fontsize=8)
    axes[0].text(i+w/2, f1m+0.2, f'{f1m:.1f}', ha='center', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comp_df['Model'], rotation=15, fontsize=9)
axes[0].set_ylabel('Score (%)')
axes[0].set_ylim([max(0,comp_df['F1 Macro (%)'].min()-5), 102])
axes[0].set_title('All Models — Accuracy & F1 Macro', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, comp_df['Infer (ms/smpl)'], color=cols, alpha=0.85,
             edgecolor='black', lw=0.5)
for i, val in enumerate(comp_df['Infer (ms/smpl)']):
    axes[1].text(i, val+0.0001, f'{val:.4f}ms', ha='center', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(comp_df['Model'], rotation=15, fontsize=9)
axes[1].set_ylabel('Inference Time (ms per sample)')
axes[1].set_title('Inference Speed Comparison\n(lower = faster, better for real-time)',
                   fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('All Models Comparison — Temperature Fault Detection',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_06_model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_06_model_comparison.png')

---
## 9. Final Summary

In [ ]:
print('=' * 65)
print('  PHASE 5 COMPLETE — FULL EVALUATION REPORT')
print('=' * 65)
print(f'  Dataset          : PVS1-PVS9, Left + Right temperature sensor')
print(f'  Sensor           : MPU-9250 onboard temperature')
print(f'  Fault classes    : normal, noise_fault, bias_fault, flatline_fault')
print(f'  Window size      : {p2["WINDOW_SIZE"]} samples ({p2["WINDOW_SIZE"]/p2["SAMPLING_HZ"]:.1f} sec)')
print(f'  Features used    : {len(FINAL_FEATS)}')
print(f'  Test samples     : {len(y_test):,}')
print()
print(f'  Best model       : {BEST_NAME}')
print(f'  Test Accuracy    : {acc*100:.4f}%')
print(f'  F1 Macro         : {f1_mac*100:.4f}%')
print(f'  F1 Weighted      : {f1_wtd*100:.4f}%')
print(f'  Macro AUC        : {macro_auc:.4f}' if y_prob is not None else '')
print(f'  CV F1 (5-fold)   : {cv_f1.mean()*100:.3f}% +/- {cv_f1.std()*100:.3f}%')
print()
print('  Plots saved to:', OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.png'):
        print(f'    {f}')
print()
print('  Limitation note:')
print('  Faults were synthetically injected. Performance on real-world')
print('  hardware faults may differ. Cross-sensor expansion (accelerometer,')
print('  gyroscope) is the recommended next step.')